# 🚕 Optimización de RMSLE sin *data leakage* — taxis amarillos de Nueva York

**Reto:** entrenar modelos de regresión para estimar `trip_duration` (segundos) con `train.csv` y predecir cada `id` de `test.csv`.

**Importante:** el objetivo de **RMSLE ≈ 0.20 no se da por alcanzado**. Las métricas se miden con un conjunto etiquetado independiente de la prueba ciega. No existe una métrica verdadera disponible para `test.csv`. La selección conserva el resultado más bajo de validación y el holdout se consulta únicamente al final.

**Orden:** (1) problema y métrica → (2) auditoría y partición → (3) GPS y tiempo → (4) agrupaciones espaciales → (5) *target encoding* OOF → (6) limpieza e imputación → (7) comparación de LightGBM → (8) evaluación → (9) entrenamiento final y entrega.

Las presentaciones de *Ciclo de Vida del ML* enfatizan **evaluar el modelo con la métrica del problema** y ajustar las transformaciones únicamente en entrenamiento.

## 1. Métrica principal y objetivos realistas

Se trata de **regresión**, por lo que *accuracy*, F1, *precision* y *recall* no aplican al objetivo continuo. Se usa:

$$\mathrm{RMSLE} = \sqrt{\frac{1}{n}\sum_{i=1}^n (\ln(1+y_i)-\ln(1+\hat y_i))^2}.$$

Predecimos $z=\ln(1+y)$ y revertimos con $\hat y=\exp(\hat z)-1$. **Menor RMSLE es mejor.** 

Precaución: hay 984 viajes mayores de 12 horas, concentrados en el segundo proveedor; excluirlos de la evaluación maquillaría el resultado. Se excluyen **solo al ajustar** para reducir el efecto de etiquetas de origen anómalo.

**Anti-leakage:** aplicar cualquier `fit` de un transformador **solo** al entrenamiento; las estadísticas de `trip_duration` para las filas de ajuste se generan con KFold out-of-fold.

## 2. Librerías y funciones reproducibles

`Augment` aprende 90 zonas espaciales sobre coordenadas de entrenamiento. `LeakageFreeEncoder` calcula medias suavizadas en escala logarítmica; cada fila de ajuste se transforma usando las etiquetas de otros folds.

In [ ]:
import os,sys,time,json,traceback
os.environ['OMP_NUM_THREADS']='4'
from pathlib import Path
import pandas as pd,numpy as np, joblib
from sklearn.model_selection import train_test_split, KFold
from sklearn.impute import SimpleImputer
from sklearn.cluster import MiniBatchKMeans
from sklearn.decomposition import PCA
from lightgbm import LGBMRegressor,early_stopping,log_evaluation
from sklearn.metrics import mean_squared_log_error,mean_absolute_error
import numpy as np
import pandas as pd

def haversine_np(lat1, lon1, lat2, lon2):
    """Distancia ortodrómica en km, con NumPy vectorizado."""
    lat1,lon1,lat2,lon2=map(np.radians,(lat1,lon1,lat2,lon2))
    dl=lat2-lat1; dn=lon2-lon1
    a=np.sin(dl/2)**2+np.cos(lat1)*np.cos(lat2)*np.sin(dn/2)**2
    return 6371.0088*2*np.arcsin(np.sqrt(np.clip(a,0,1)))

def create_features(frame):
    d=frame.copy()
    out=pd.DataFrame(index=d.index)
    dt=pd.to_datetime(d['pickup_datetime'],errors='coerce')
    hour=dt.dt.hour+dt.dt.minute/60+dt.dt.second/3600
    wd=dt.dt.dayofweek
    out['vendor_id']=pd.to_numeric(d.vendor_id,errors='coerce')
    out['passenger_count']=pd.to_numeric(d.passenger_count,errors='coerce')
    out['passenger_invalid']=(out.passenger_count<1).astype('int8')
    out['store_and_fwd_flag']=(d.store_and_fwd_flag=='Y').astype('int8')
    out['month']=dt.dt.month
    out['day']=dt.dt.day
    out['day_of_year']=dt.dt.dayofyear
    out['day_of_week']=wd
    out['week_of_year']=dt.dt.isocalendar().week.astype('float64')
    out['hour']=dt.dt.hour
    out['hour_decimal']=hour
    out['minute']=dt.dt.minute
    out['is_weekend']=wd.isin([5,6]).astype('int8')
    out['is_rush_hour']=((wd<5)&(((hour>=7)&(hour<10))|((hour>=16)&(hour<20)))).astype('int8')
    out['is_night']=((hour<6)|(hour>=22)).astype('int8')
    out['hour_sin']=np.sin(2*np.pi*hour/24)
    out['hour_cos']=np.cos(2*np.pi*hour/24)
    out['weekday_sin']=np.sin(2*np.pi*wd/7)
    out['weekday_cos']=np.cos(2*np.pi*wd/7)
    out['dayyear_sin']=np.sin(2*np.pi*dt.dt.dayofyear/366)
    out['dayyear_cos']=np.cos(2*np.pi*dt.dt.dayofyear/366)
    out['month_sin']=np.sin(2*np.pi*(dt.dt.month-1)/12)
    out['month_cos']=np.cos(2*np.pi*(dt.dt.month-1)/12)
    for c in ['pickup_longitude','pickup_latitude','dropoff_longitude','dropoff_latitude']:
        out[c]=pd.to_numeric(d[c],errors='coerce')
    for prefix in ['pickup','dropoff']:
        lon=out[f'{prefix}_longitude']; lat=out[f'{prefix}_latitude']
        bad=(~lon.between(-74.5,-73.3))|(~lat.between(40.4,41.2))
        out[f'{prefix}_bad_gps']=bad.astype('int8')
        out.loc[bad,[f'{prefix}_longitude',f'{prefix}_latitude']]=np.nan
    lon1=out.pickup_longitude.to_numpy(); lon2=out.dropoff_longitude.to_numpy()
    lat1=out.pickup_latitude.to_numpy(); lat2=out.dropoff_latitude.to_numpy()
    latm=(lat1+lat2)/2
    dy=(lat2-lat1)*111.195
    dx=(lon2-lon1)*111.195*np.cos(np.radians(latm))
    out['dx_km']=dx; out['dy_km']=dy
    out['abs_dx_km']=np.abs(dx);out['abs_dy_km']=np.abs(dy)
    out['haversine_km']=haversine_np(lat1,lon1,lat2,lon2)
    out['manhattan_km']=np.abs(dx)+np.abs(dy)
    out['haversine_log']=np.log1p(out['haversine_km'])
    out['manhattan_log']=np.log1p(out['manhattan_km'])
    out['trip_distance_squared']=out.haversine_km**2
    out['manhattan_ratio']=out.manhattan_km/(out.haversine_km+0.1)
    out['is_near_zero_distance']=(out.haversine_km<0.10).astype('int8')
    out['bearing']=np.arctan2(dx,dy)
    out['bearing_sin']=np.sin(out.bearing)
    out['bearing_cos']=np.cos(out.bearing)
    out['mid_lat']=(lat1+lat2)/2
    out['mid_lon']=(lon1+lon2)/2
    out['lat_lon_start_prod']=lat1*lon1
    out['lat_lon_end_prod']=lat2*lon2
    for prefix in ['pickup','dropoff']:
        lon=out[f'{prefix}_longitude'];lat=out[f'{prefix}_latitude']
        out[f'{prefix}_grid_lat_100']=(lat*100).round()
        out[f'{prefix}_grid_lon_100']=(lon*100).round()
        out[f'{prefix}_grid_lat_200']=(lat*200).round()
        out[f'{prefix}_grid_lon_200']=(lon*200).round()
        out[f'{prefix}_grid_lat_500']=(lat*500).round()
        out[f'{prefix}_grid_lon_500']=(lon*500).round()
        # Las ubicaciones son geografía pública, no se usan destinos posteriores desconocidos.
        for name,alat,alon in [('jfk',40.6413,-73.7781),('lga',40.7769,-73.8740),
                               ('ewr',40.6895,-74.1745),('times_sq',40.7580,-73.9855),
                               ('grand_central',40.7527,-73.9772),('downtown',40.7061,-74.0086)]:
            out[f'{prefix}_dist_{name}_km']=haversine_np(lat.to_numpy(),lon.to_numpy(),alat,alon)
        out[f'{prefix}_near_jfk']=(out[f'{prefix}_dist_jfk_km']<2.5).astype('int8')
        out[f'{prefix}_near_lga']=(out[f'{prefix}_dist_lga_km']<2.5).astype('int8')
        out[f'{prefix}_near_ewr']=(out[f'{prefix}_dist_ewr_km']<3.5).astype('int8')
    out['any_airport']=(out[['pickup_near_jfk','pickup_near_lga','pickup_near_ewr','dropoff_near_jfk','dropoff_near_lga','dropoff_near_ewr']].max(axis=1))
    out['hour_x_distance']=out.hour_decimal*out.haversine_km
    out['rush_x_distance']=out.is_rush_hour*out.haversine_km
    out['weekend_x_distance']=out.is_weekend*out.haversine_km
    # Reducir memoria y normalizar la representación numérica (NO estandarizar estadísticos).
    return out.astype(np.float32).replace([np.inf,-np.inf],np.nan)

R=Path.cwd(); out=R/'mejora_taxis'/'v2';out.mkdir(parents=True,exist_ok=True)
t0=time.time()
def log(*xs): print(round(time.time()-t0,1),'s',*xs,flush=True)
def rmsle(y,p):return float(np.sqrt(mean_squared_log_error(y,np.maximum(p,0))))
def latlon_to_xy(lat,lon):return np.column_stack(((lon+73.99)*84.0,(lat-40.75)*111.2)).astype(np.float32)
def cluster_xy(fr,col):
    a=latlon_to_xy(fr[f'{col}_latitude'],fr[f'{col}_longitude']);return np.nan_to_num(a,nan=-100.,posinf=-100.,neginf=-100.)

class Augment:
 def __init__(self,n_clusters=90):self.n_clusters=n_clusters
 def fit(self,fr):
    both=np.concatenate([cluster_xy(fr,'pickup'),cluster_xy(fr,'dropoff')],axis=0)
    rng=np.random.default_rng(42);ix=rng.choice(len(both),min(175000,len(both)),replace=False)
    self.cluster=MiniBatchKMeans(n_clusters=self.n_clusters,n_init=2,batch_size=8192,max_iter=125,random_state=42).fit(both[ix])
    return self
 def transform(self,fr):
    op=cluster_xy(fr,'pickup');de=cluster_xy(fr,'dropoff');a=pd.DataFrame(index=fr.index)
    a['pickup_cluster']=self.cluster.predict(op).astype(np.int16)
    a['dropoff_cluster']=self.cluster.predict(de).astype(np.int16)
    a['cluster_pair']=(a.pickup_cluster.astype('int32')*self.n_clusters+a.dropoff_cluster).astype('int32')
    # Natural path grid with 200-400m cells.
    for prefix,xy in [('pickup',op),('dropoff',de)]:
        a[f'{prefix}_x']=xy[:,0]
        a[f'{prefix}_y']=xy[:,1]
        a[f'{prefix}_axis']=xy[:,1]-xy[:,0]*.25
        # Manhattan: in NYC 29 degree rotated grid coordinate; geometry deterministic.
        a[f'{prefix}_manh_a']=xy[:,0]*.87+xy[:,1]*.49
        a[f'{prefix}_manh_b']=-xy[:,0]*.49+xy[:,1]*.87
        gx=np.clip(np.floor((xy[:,0]+35)/.45).astype(np.int32),0,999)
        gy=np.clip(np.floor((xy[:,1]+40)/.45).astype(np.int32),0,999)
        a[f'{prefix}_grid_code']=(gx*1000+gy).astype(np.int32)
    dt=pd.to_datetime(fr.pickup_datetime)
    a['hour_bucket']=(dt.dt.hour//3).astype(np.int8)
    a['weekday_bucket']=(dt.dt.dayofweek).astype(np.int8)
    a['cluster_hour']=(a.pickup_cluster*8+a.hour_bucket).astype(np.int32)
    a['pickup_day_hour']=(a.pickup_cluster*56+a.weekday_bucket*8+a.hour_bucket).astype(np.int32)
    a['route_hour']=(a.cluster_pair*8+a.hour_bucket).astype(np.int32)
    a['od_manhattan_axis_delta']=a.dropoff_manh_a-a.pickup_manh_a
    a['od_manhattan_cross_delta']=a.dropoff_manh_b-a.pickup_manh_b
    a['od_manhattan_axis_abs']=np.abs(a.od_manhattan_axis_delta)
    a['od_manhattan_cross_abs']=np.abs(a.od_manhattan_cross_delta)
    # Complements for route/varying congestion.
    a['manh_axis_distance']=np.abs(a.od_manhattan_axis_delta)+np.abs(a.od_manhattan_cross_delta)
    return a

class LeakageFreeEncoder:
 def __init__(self,keys=['cluster_pair','cluster_hour','pickup_day_hour','route_hour'], smooth=65, folds=5):self.keys=keys;self.smooth=smooth;self.folds=folds
 def _map(self,train_f,target,out_f,key):
    df=pd.DataFrame({'k':train_f[key].values,'y':target})
    st=df.groupby('k',sort=False).y.agg(['mean','count'])
    mu=np.mean(target)
    values=(st['mean']*st['count']+mu*self.smooth)/(st['count']+self.smooth)
    pred=out_f[key].map(values).fillna(mu).to_numpy(dtype=np.float32)
    cnt=out_f[key].map(st['count']).fillna(0).to_numpy(dtype=np.float32)
    return pred,np.log1p(cnt)
 def fit_transform(self,fr,logtarget):
    results={k:np.empty(len(fr),dtype=np.float32) for k in self.keys}
    counts={k:np.empty(len(fr),dtype=np.float32) for k in self.keys}
    kfold=KFold(n_splits=self.folds,shuffle=True,random_state=47)
    for it,iv in kfold.split(fr):
        for k in self.keys:
            results[k][iv],counts[k][iv]=self._map(fr.iloc[it],logtarget[it],fr.iloc[iv],k)
    return pd.DataFrame({**{f'oof_mean_{k}':v for k,v in results.items()},**{f'log_n_{k}':v for k,v in counts.items()}},index=fr.index)
 def fit(self,fr,logtarget):self.ref=fr.reset_index(drop=True);self.logtarget=np.asarray(logtarget);return self
 def transform(self,fr):
    res={}
    for k in self.keys:
        preds,counts=self._map(self.ref,self.logtarget,fr,k)
        res[f'oof_mean_{k}']=preds;res[f'log_n_{k}']=counts
    return pd.DataFrame(res,index=fr.index)


## 3. Carga de datos, limpieza inicial y partición

Las mismas semillas (`42`) permiten comparar con el proyecto anterior. Se apartan 70,000 viajes para validación y 70,000 para evaluación final; los restantes se usan para ajustar el modelo. Los viajes superiores a 12 horas solo se eliminan **del entrenamiento**, nunca de validación o holdout.

In [ ]:
log('Cargando datos')
df=pd.read_csv(R/'train.csv',parse_dates=['pickup_datetime']); idx=np.arange(len(df))
itr,itemp=train_test_split(idx,test_size=.20,random_state=42)
iva,iho=train_test_split(itemp,test_size=.50,random_state=42)
tr=df.iloc[itr].copy();va=df.iloc[iva].copy();ho=df.iloc[iho].copy()
tr=tr.loc[tr.trip_duration.between(1,43200)].copy()
ytr=tr.trip_duration.to_numpy(dtype=np.float64);ztr=np.log1p(ytr)
yva=va.trip_duration.to_numpy(dtype=np.float64);zva=np.log1p(yva)
log('Partición',len(tr),len(va),len(ho))


## 4. Ingeniería geográfica y temporal

Las 81 variables originales se conservan. Se incorporan ejes de Manhattan, coordenadas métricas locales y distancias orientadas. Todas son calculables con el origen, destino y momento en que empieza el viaje; no se utiliza hora de llegada ni duración de viaje.

In [ ]:
aug=Augment(90).fit(tr)
feats={}
for name,part in [('train',tr),('valid',va),('holdout',ho)]:
   log('Creando características',name)
   base=create_features(part).reset_index(drop=True)
   geo=aug.transform(part).reset_index(drop=True)
   feats[name]=pd.concat([base,geo],axis=1)
gtrain=feats['train'];gvalid=feats['valid'];ghold=feats['holdout']


## 5. *Target encoding* out-of-fold sin fuga

Dividimos entrenamiento en cinco folds. Para cada fold, calculamos medias de `log1p(trip_duration)` y frecuencias de grupos de zonas/horas con **los otros cuatro**. Validación y holdout solo reciben estadísticas calculadas con el entrenamiento etiquetado. Las medias tienen regularización (*smoothing*=80) para evitar sobreajustar grupos raros.

Después imputamos los valores no válidos con la mediana **aprendida solo de entrenamiento**. No escalamos LightGBM porque los árboles separan variables por umbrales; la normalización sería obligatoria para ciertas alternativas lineales/neuronales, no para estos árboles.

In [ ]:
catcols=['pickup_cluster','dropoff_cluster','cluster_pair','pickup_grid_code','dropoff_grid_code','cluster_hour','pickup_day_hour','route_hour']
log('Codificando sin fuga: OOF entrenamiento, mapa completo solo validación')
enc=LeakageFreeEncoder(smooth=80,folds=5)
enc_train=enc.fit_transform(gtrain,ztr)
enc.fit(gtrain,ztr)
enc_valid=enc.transform(gvalid)
enc_hold=enc.transform(ghold)
Xtr=pd.concat([gtrain,enc_train],axis=1)
Xva=pd.concat([gvalid,enc_valid],axis=1)
Xho=pd.concat([ghold,enc_hold],axis=1)
Xcols=list(Xtr.columns)
imp=SimpleImputer(strategy='median',keep_empty_features=True)
Xtr=pd.DataFrame(imp.fit_transform(Xtr).astype(np.float32),columns=Xcols)
Xva=pd.DataFrame(imp.transform(Xva[Xcols]).astype(np.float32),columns=Xcols)
Xho=pd.DataFrame(imp.transform(Xho[Xcols]).astype(np.float32),columns=Xcols)


## 6. Selección de predictores y parámetros comparables

**Primer intento descartado:** marcar cientos de zonas como categorías en LightGBM produjo sobreajuste (RMSLE ≈ 0.392–0.394). En consecuencia, las nuevas variantes conservan únicamente 15 variables geográficas continuas, y opcionalmente las estadísticas OOF. Todos los modelos usan hiperparámetros comparables.

In [ ]:
# Conservamos de las nuevas variables solamente las continuas y geométricas,
# evitando códigos de categorías muy específicos que sobreajustaron el primer intento.
cols_geo=['pickup_x','pickup_y','pickup_axis','pickup_manh_a','pickup_manh_b',
          'dropoff_x','dropoff_y','dropoff_axis','dropoff_manh_a','dropoff_manh_b',
          'od_manhattan_axis_delta','od_manhattan_cross_delta',
          'od_manhattan_axis_abs','od_manhattan_cross_abs','manh_axis_distance']
orig_cols=list(create_features(tr.iloc[:1]).columns)
params={'n_estimators':1600,'learning_rate':.045,'num_leaves':105,
        'max_depth':-1,'min_child_samples':75,'colsample_bytree':.92,
        'subsample':.9,'subsample_freq':1,'reg_lambda':5}
opts=[('referencia_81',False,params),('rotacion_urbana',False,params),
      ('rotacion_mas_oof',True,params)]


## 7. Entrenamiento, parada anticipada y RMSLE de validación

El objetivo de LightGBM es `regression` sobre $z=\log(1+y)$; su RMSE de validación es algebraicamente igual al RMSLE de la predicción invertida (sin efectos de recorte). La parada anticipada se decide **solo con validación**, nunca con los IDs de prueba ciega.

In [ ]:
hist=[];zpreds={}
for name,uses_oof,params in opts:
   if name=='referencia_81': cols=orig_cols
   elif not uses_oof: cols=orig_cols+cols_geo
   else: cols=orig_cols+cols_geo+[c for c in Xcols if c.startswith(('oof_','log_n_'))]
   log('Entrenando',name,'columnas',len(cols))
   m=LGBMRegressor(**params,objective='regression',random_state=42,n_jobs=4,verbosity=-1)
   m.fit(Xtr[cols],ztr,eval_set=[(Xva[cols],zva)],eval_metric='rmse',callbacks=[early_stopping(100,verbose=False)],categorical_feature=[])
   p=m.predict(Xva[cols]);real=np.expm1(np.maximum(p,0))
   score=rmsle(yva,real)
   hist.append({'modelo':name,'rmsle_validacion':score,'mejor_iteracion':int(m.best_iteration_),'n_features':len(cols),'segundos':round(time.time()-t0,1)})
   zpreds[name]=p
   log(name,'RMSLE',score,'iter',m.best_iteration_)
   pd.DataFrame(hist).to_csv(out/'experimentos.csv',index=False)
   if name=='rotacion_mas_oof':
       log('guardando mapa de importancias')
       pd.DataFrame({'feature':cols,'gain':m.booster_.feature_importance(importance_type='gain')}).sort_values('gain',ascending=False).to_csv(out/'importancia_geo_oof.csv',index=False)
   if True:
       joblib.dump({'model':m,'cols':cols,'augment':aug,'encoder':enc,'imputer':imp},out/(name+'_validacion.joblib'),compress=2)


## 8. Ensambles y selección (exclusivamente en validación)

Se combinan resultados en escala logarítmica antes de transformar a segundos. Los pesos del ensamble se comparan **únicamente** en validación. Un ensamble se usa solamente si mejora realmente.

In [ ]:
log('Validando combinaciones en escala logarítmica')
for base,other in [('referencia_81','rotacion_urbana'),('referencia_81','rotacion_mas_oof')]:
   for w in [0.25,0.5,0.75]:
       zz=w*zpreds[base]+(1-w)*zpreds[other]
       sc=rmsle(yva,np.expm1(np.maximum(0,zz)))
       hist.append({'modelo':f'blend_{base}_{other}_{w}','rmsle_validacion':sc,'mejor_iteracion':0,'n_features':len(Xcols)})
pd.DataFrame(hist).sort_values('rmsle_validacion').to_csv(out/'experimentos.csv',index=False)


In [ ]:
log('Ranking',sorted([(r['modelo'],round(r['rmsle_validacion'],6)) for r in hist],key=lambda x:x[1]))


## 9. Evaluación final en holdout

La partición de 70,000 viajes no se usa para entrenar, calcular estadísticas OOF ni ajustar pesos. Se evalúa **una sola vez tras seleccionar**. Hay que reportar el RMSLE original y nuevo, sin prometer que el score de la plataforma será igual.

In [ ]:
# Holdout only winner once; do not use it for tuning.
top=sorted(hist,key=lambda x:x['rmsle_validacion'])[0]
name=top['modelo'];log('Ganador',name)
if name.startswith('blend_'):
   assert '_rotacion_' in name
   w=float(name.rsplit('_',1)[-1])
   other='rotacion_urbana' if 'rotacion_urbana' in name else 'rotacion_mas_oof'
   comps=[('referencia_81',w),(other,1-w)]
else: comps=[(name,1)]
pp=0
for nn,w in comps:
   pack=joblib.load(out/(nn+'_validacion.joblib'))
   if pack is None: raise RuntimeError('Ganador inesperado sin modelo serializado')
   pp+=w*pack['model'].predict(Xho[pack['cols']])
ppho=np.expm1(np.maximum(0,pp));score_ho=rmsle(ho.trip_duration,ppho)
log('Holdout (una sola medición)',score_ho)
json.dump({'ganador':top,'holdout_rmsle':score_ho,'resultados':hist,'train_n':len(tr),'valid_n':len(va),'holdout_n':len(ho),'semilla':42},open(out/'metricas_mejora.json','w'),indent=2)
log('EXPERIMENTO COMPLETO')


## 10. Reajuste final y `submission.csv`

Tras decidir el ganador, **ahora sí** utilizamos todas las etiquetas conocidas de `train.csv` para el ajuste final, sin incorporar etiquetas del CSV ciego. Para el ajuste final, las estadísticas OOF se recalculan otra vez en cinco folds. Los transformadores de test usan solo `transform`, nunca `fit`. 

Esta celda crea `submission_mejora.csv` con **exactamente** las columnas `id` y `trip_duration`, una fila por ID, sin negativos. Si el resultado de validación y holdout NO supera el modelo anterior, no reemplaces tu entrega vigente.

In [ ]:
"""Reentrenamiento final de una mejora MODESTA, sin leakage: OOF mediante bincount."""
import json,time,gc
from pathlib import Path
import numpy as np,pandas as pd,joblib
from lightgbm import LGBMRegressor
from sklearn.model_selection import KFold
from sklearn.impute import SimpleImputer


R=Path.cwd();O=R/'mejora_taxis';O.mkdir(exist_ok=True)
KEYS=['cluster_pair','cluster_hour','pickup_day_hour','route_hour'];SMOOTH=80.
GEO=['pickup_x','pickup_y','pickup_axis','pickup_manh_a','pickup_manh_b','dropoff_x','dropoff_y','dropoff_axis','dropoff_manh_a','dropoff_manh_b','od_manhattan_axis_delta','od_manhattan_cross_delta','od_manhattan_axis_abs','od_manhattan_cross_abs','manh_axis_distance']

def encode_oof_and_maps(train_keys,z,test_keys):
    n=len(z);kf=KFold(n_splits=5,shuffle=True,random_state=47)
    result={};ptest={};pack={};mu_all=z.mean()
    for name in KEYS:
        values=train_keys[name].to_numpy(dtype=np.int32,copy=True)
        unseen=test_keys[name].to_numpy(dtype=np.int32,copy=True)
        size=int(max(values.max(),unseen.max()))+1
        oof=np.zeros(n,dtype=np.float32);ocnt=np.zeros(n,dtype=np.float32)
        for tr,va in kf.split(np.arange(n)):
            counts=np.bincount(values[tr],minlength=size)
            sums=np.bincount(values[tr],weights=z[tr],minlength=size)
            mu=z[tr].mean()
            means=(sums+mu*SMOOTH)/(counts+SMOOTH)
            oof[va]=means[values[va]].astype(np.float32)
            ocnt[va]=np.log1p(counts[values[va]]).astype(np.float32)
        counts=np.bincount(values,minlength=size)
        sums=np.bincount(values,weights=z,minlength=size)
        mean_full=((sums+mu_all*SMOOTH)/(counts+SMOOTH)).astype(np.float32)
        cnt_full=np.log1p(counts).astype(np.float32)
        result['oof_mean_'+name]=oof;result['log_n_'+name]=ocnt
        ptest['oof_mean_'+name]=mean_full[unseen]
        ptest['log_n_'+name]=cnt_full[unseen]
        pack[name]={'mean':mean_full,'log_count':cnt_full}
        print('  OOF',name,'size',size,flush=True)
    return pd.DataFrame(result),pd.DataFrame(ptest),{'means':pack,'keys':KEYS}

def entrenar_final():
    t=time.time();print('Cargando train/test',flush=True)
    full=pd.read_csv(R/'train.csv',parse_dates=['pickup_datetime'])
    test=pd.read_csv(R/'test.csv',parse_dates=['pickup_datetime'])
    assert full.id.is_unique and test.id.is_unique and len(test)==100000
    full=full.loc[full.trip_duration.between(1,43200)].copy()
    z=np.log1p(full.trip_duration.to_numpy(dtype=np.float64))
    aug=Augment(90).fit(full)
    bgfull=create_features(full).reset_index(drop=True)
    bgtest=create_features(test).reset_index(drop=True)
    geo_full=aug.transform(full).reset_index(drop=True)
    geo_test=aug.transform(test).reset_index(drop=True)
    print('Ingeniería terminada',round(time.time()-t,1),'s',flush=True)
    oof,ptest,compact_maps=encode_oof_and_maps(geo_full[KEYS],z,geo_test[KEYS])
    train_matrix=pd.concat([bgfull,geo_full[GEO],oof],axis=1)
    test_matrix=pd.concat([bgtest,geo_test[GEO],ptest],axis=1)
    columns=list(train_matrix.columns);orig_cols=list(bgfull.columns)
    del bgfull,bgtest,geo_full,geo_test,oof,ptest,full;gc.collect()
    imp=SimpleImputer(strategy='median',keep_empty_features=True)
    arr=imp.fit_transform(train_matrix).astype(np.float32,copy=False)
    arr_test=imp.transform(test_matrix[columns]).astype(np.float32,copy=False)
    del train_matrix,test_matrix;gc.collect()
    train_matrix=pd.DataFrame(arr,columns=columns,copy=False)
    test_matrix=pd.DataFrame(arr_test,columns=columns,copy=False)
    orig=orig_cols
    xlist=[('referencia_81',.5,orig),('rotacion_mas_oof',.5,columns)]
    params={'n_estimators':1600,'learning_rate':.045,'num_leaves':105,'max_depth':-1,'min_child_samples':75,'colsample_bytree':.92,'subsample':.9,'subsample_freq':1,'reg_lambda':5,'objective':'regression','random_state':42,'n_jobs':4,'verbosity':-1}
    log_pred=np.zeros(len(test),dtype=np.float64)
    models=[]
    for name,w,cols in xlist:
        print('Entrenando',name,'features',len(cols),'s',round(time.time()-t,1),flush=True)
        model=LGBMRegressor(**params).fit(train_matrix[cols],z)
        log_pred+=w*model.predict(test_matrix[cols])
        model.booster_.save_model(str(O/(name+'_final.txt')))
        models.append({'nombre':name,'peso':w,'columnas':cols,'modelo':model})
        gc.collect()
    preds=np.expm1(np.maximum(log_pred,0))
    submission=pd.DataFrame({'id':test.id.to_numpy(),'trip_duration':preds})
    assert submission.columns.tolist()==['id','trip_duration']
    assert len(submission)==len(test)==100000
    assert submission.id.is_unique and (submission.trip_duration>=0).all() and np.isfinite(submission.trip_duration).all()
    subfile=O/'submission_experimental.csv';submission.to_csv(subfile,index=False)
    assert np.array_equal(pd.read_csv(subfile)['id'].to_numpy(),test['id'].to_numpy())
    pack={'modelos':models,'imputer':imp,'augment':aug,'encoder_maps':compact_maps,'columnas_todas':columns,'columnas_geo':GEO}
    joblib.dump(pack,O/'modelo_compacto.joblib',compress=3)
    meta={'filas_entrenadas':len(z),'test_filas':len(test),'pesos':{m['nombre']:m['peso'] for m in models},'rmsle_validacion':0.3746240172258088,'rmsle_holdout':0.37928817537192905,'nota':'La métrica de test.csv ciego no se conoce. No está garantizada mejora sobre 0.3704 externo.','duracion_s':round(time.time()-t,1)}
    (O/'meta_final.json').write_text(json.dumps(meta,indent=2,ensure_ascii=False),encoding='utf8')
    print('Listo',subfile, 'MIN/MAX/MED',float(preds.min()),float(preds.max()),float(np.median(preds)),'s',round(time.time()-t,1),flush=True)
entrenar_final()


## 11. Conclusión y rigor académico

El objetivo es **minimizar RMSLE sin contaminar la validación**. La prueba ciega no tiene etiquetas: su rendimiento real será conocido únicamente al evaluarla externamente. *No se debe afirmar un RMSLE de 0.20 sin obtenerlo en una evaluación fiable*. Tampoco debe reemplazarse un modelo superior por uno experimental que no lo superó.

Los intentos que empeoran el resultado son útiles para justificar por qué se descartan ciertas variables, pero no constituyen mejoras.

**Resultados medidos de esta comparación:** RMSLE validación = **0.374624**, RMSLE holdout = **0.379288**. Estos resultados no implican que el CSV ciego obtenga el mismo RMSLE ni mejoren un score externo de 0.3704. La parte final reentrena únicamente después de haber elegido el modelo con validación.
